In [2]:
import sys
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim, regexp_replace
from delta import *

# ===================================================
# 1. CONFIGURAÇÃO DO AMBIENTE (SPARK + DELTA)
# ===================================================
print("🚀 A iniciar Spark Session...")

warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession.builder \
    .appName("Silver ETL - Netflix Consolidated") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("✅ Sessão iniciada.")

# ===================================================
# 2. UPLOAD DO FICHEIRO (Segurança)
# ===================================================
# Verifica se o ficheiro já está no Bronze, senão tenta fazer upload
try:
    from hdfs import InsecureClient
    client = InsecureClient("http://hdfs-nn:9870", user="anonymous")
    
    # Caminho onde o CSV deve estar no TEU computador
    local_csv = "titles_netflix.csv" 
    bronze_path = "/demo/bronze/titles_netflix.csv"
    
    if os.path.exists(local_csv):
        print(f"⬆️ A enviar {local_csv} para o HDFS...")
        client.upload(bronze_path, local_csv, overwrite=True)
        print("✅ Upload concluído.")
    else:
        print(f"⚠️ O ficheiro local '{local_csv}' não foi encontrado. A assumir que já está no HDFS.")
except ImportError:
    print("⚠️ Biblioteca 'hdfs' não instalada ou erro de conexão. A saltar upload.")

# ===================================================
# 3. LEITURA E LIMPEZA (Baseado nos Notebooks #4 e #5)
# ===================================================
hdfs_source = "hdfs://hdfs-nn:9000/demo/bronze/titles_netflix.csv"
print(f"\n📖 A ler Bronze de: {hdfs_source}")

try:
    df_raw = spark.read.csv(
        hdfs_source,
        header=True,
        inferSchema=True,
        multiLine=True, # Importante para descrições longas
        escape='"'
    )
except Exception as e:
    print(f"❌ Erro ao ler CSV: {e}")
    # Se der erro, para o script aqui
    raise e

# --- TRANSFORMAÇÕES ---
print("🧹 A aplicar limpezas...")

# 1. Correção de Tipos (seasons e scores vêm muitas vezes como string)
df_clean = df_raw \
    .withColumn("seasons", col("seasons").cast("int")) \
    .withColumn("imdb_score", col("imdb_score").cast("double")) \
    .withColumn("tmdb_score", col("tmdb_score").cast("double")) \
    .withColumn("release_year", col("release_year").cast("int"))

# 2. Tratamento de Nulos (Lógica do Notebook #5)
# Substitui nulos por 'Unknown' em colunas de texto
cols_texto = ["title", "type", "description", "age_certification", "genres", "production_countries"]
df_clean = df_clean.fillna("Unknown", subset=cols_texto)

# Substitui nulos por 0 em colunas numéricas
cols_num = ["runtime", "seasons", "imdb_score", "imdb_votes", "tmdb_popularity"]
df_clean = df_clean.fillna(0, subset=cols_num)

# 3. Limpeza de Caracteres Estranhos (Opcional, mas estava no notebook do teu amigo)
# Remove parênteses retos [] dos géneros e países para ficar mais limpo
# Ex: "['Drama', 'Comedy']" -> "Drama, Comedy"
df_clean = df_clean \
    .withColumn("genres", regexp_replace(col("genres"), r"[\[\]']", "")) \
    .withColumn("production_countries", regexp_replace(col("production_countries"), r"[\[\]']", ""))

print("✅ Dados limpos.")

# ===================================================
# 4. GRAVAÇÃO SILVER (DELTA)
# ===================================================
# Caminho alinhado com a tua estrutura: /warehouse/silver.db/netflix_titles
target_path = "hdfs://hdfs-nn:9000/warehouse/silver.db/netflix_titles"
print(f"\n💾 A gravar Tabela Delta em: {target_path}")

# Apaga a tabela antiga se existir (para garantir que fica limpa)
spark.sql("DROP TABLE IF EXISTS silver.netflix_titles")

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("release_year") \
    .save(target_path)

# ===================================================
# 5. REGISTO NO HIVE (CORRIGIDO)
# ===================================================
print("📝 A registar no catálogo Hive...")

spark.sql("CREATE DATABASE IF NOT EXISTS silver LOCATION 'hdfs://hdfs-nn:9000/warehouse/silver.db'")

# Criar a tabela a apontar para os dados Delta
# Nota: Como é Delta, não precisamos de definir esquema nem partições aqui, ele lê dos ficheiros!
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS silver.netflix_titles
    USING DELTA
    LOCATION '{target_path}'
""")

# --- LINHA REMOVIDA: spark.catalog.recoverPartitions(...) ---
# O Delta faz isto automaticamente, por isso dava erro ao forçar.

print("\n🎉 SUCESSO! A tabela 'silver.netflix_titles' foi criada.")
print("--- Amostra dos Dados ---")
spark.sql("SELECT title, release_year, genres, imdb_score FROM silver.netflix_titles LIMIT 5").show(truncate=False)

🚀 A iniciar Spark Session...
✅ Sessão iniciada.
⚠️ O ficheiro local 'titles_netflix.csv' não foi encontrado. A assumir que já está no HDFS.

📖 A ler Bronze de: hdfs://hdfs-nn:9000/demo/bronze/titles_netflix.csv
🧹 A aplicar limpezas...
✅ Dados limpos.

💾 A gravar Tabela Delta em: hdfs://hdfs-nn:9000/warehouse/silver.db/netflix_titles
📝 A registar no catálogo Hive...

🎉 SUCESSO! A tabela 'silver.netflix_titles' foi criada.
--- Amostra dos Dados ---
+-------------------+------------+-----------------------------------+----------+
|title              |release_year|genres                             |imdb_score|
+-------------------+------------+-----------------------------------+----------+
|Power Rangers      |1993        |action, scifi, fantasy, family     |6.5       |
|Last Action Hero   |1993        |action, fantasy, comedy, family    |6.4       |
|In the Line of Fire|1993        |drama, thriller, crime, action     |7.2       |
|Cliffhanger        |1993        |action, thriller, comed